In [18]:
import ollama
from loguru import logger
import os
import requests
from dotenv import load_dotenv

load_dotenv()

OLLAMA_API_KEY = os.getenv("WEB_SEARCH_API_KEY")

if not OLLAMA_API_KEY:
    raise ValueError("OLLAMA_API_KEY not found in environment variables.")

In [19]:
from utils.logger import setup_logger

logger = setup_logger()

## Define web-search tool

In [20]:
def web_search(query: str, max_results: int = 1) -> dict:
    """ Perform a web search using the Ollama API.
    
    Args:
        query (str): The search query.
        max_results (int): The maximum number of results to return.
    
    Returns:
        dict: The search results from the API.
    """
    url = "https://ollama.com/api/web_search"
    headers = {
    "Authorization": f"Bearer {OLLAMA_API_KEY}",
    "Content-Type": "application/json"
    }
    # Construct the payload for the API request
    payload = {
        "query": query,
        "limit": max_results
    }
    try:
        # Make the POST request to the Ollama API
        response = requests.post(url, 
                                 headers=headers, 
                                 json=payload)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        return None

### Basic Memory Mechanisms

* **sliding window**: keep only recent messages

* **summary**: use llm to summarize key facts

In [21]:
class SummaryLLM:

    def get_summary(self, state_memory):
        summary_prompt = [
            {"role": "system", "content": "You are a helpful assistant that summarizes the conversation history to keep it concise while retaining important information."},
            {"role": "user", "content": "Summarize the following conversation history in a concise manner, keeping important details:\n" + "\n".join([f"{msg['role']}: {msg['content']}" for msg in state_memory])}
        ]
        response = ollama.chat(
            model="gpt-oss:20b",
            messages=summary_prompt,
            options={"temperature": 0}
        )
        summary = response["message"].get("content", "")
        return summary
    
class WebContentSummaryLLM:

    def summarize_content(self, query, content):
        summary_prompt = [
            {"role": "system", "content": "You are a helpful assistant that summarizes web search results to extract key information."},
            {"role": "user", "content": f"Summarize the following content in a concise manner, extracting key information relevant to the query: '{query}'\nContent:\n{content}"}
        ]
        response = ollama.chat(
            model="gpt-oss:20b",
            messages=summary_prompt,
            options={"temperature": 0}
        )
        summary = response["message"].get("content", "")
        return summary

In [22]:
class StatefullAgent:
    
    AVAILABLE_MEMORY_MODES = ["sliding_window", "summary"]

    def __init__(self, llm_model="gpt-oss:20b", system="", temperature=0, max_iterations=10, memory_mode="summary", window_size=10):

        if system and not isinstance(system, str):
            raise ValueError("System message must be a string.")
        if not isinstance(max_iterations, int) or max_iterations <= 0:
            raise ValueError("max_iterations must be a positive integer.")
        if memory_mode not in self.AVAILABLE_MEMORY_MODES:
            raise ValueError(f"Unknown memory_mode '{memory_mode}'. Choose from {self.AVAILABLE_MEMORY_MODES}")
        if memory_mode == "sliding_window":
            assert window_size > 0, "window_size must be a positive integer for sliding_window memory mode."

        self.llm_model = llm_model
        self.system = system
        self.temperature = temperature
        self.state_memory = []
        self.tools_used = []
        self.max_iterations = max_iterations
        if self.system:
            self.system_message = {"role": "system", "content": self.system}
        self.state_memory = [self.system_message]
        self.window_size = window_size
        self.memory_mode = memory_mode
        self.llm_summary = SummaryLLM()
        self.content_summary = WebContentSummaryLLM()

    def __call__(self, message):
        try:
            self.message = message
            self.state_memory.append({"role": "user", "content": message})
            result = self.execute()
            self.state_memory.append({"role": "assistant", "content": result})
            return result
        except Exception as e:
            logger.error(f"Error during agent call: {e}")
            return "An error occurred while processing your request."

    def get_sliding_window_memory(self):
        self.state_memory = self.state_memory[-self.window_size+1:]
        if self.system_message not in self.state_memory:
            self.state_memory.insert(0, self.system_message)

    def get_summary_memory(self):
        summary = self.llm_summary.get_summary(state_memory=self.state_memory)
        self.state_memory = [self.system_message, {"role": "assistant", "content": f"Summary of conversation so far: {summary}"}]

    def get_content_summary(self, content):
        summary = self.content_summary.summarize_content(query=self.message, content=content)
        return summary

    def execute(self):

        iterations = 0

        while iterations < self.max_iterations:

            iterations += 1

            logger_iter = logger.bind(iteration=iterations, 
                                      memory_length=len(self.state_memory), 
                                      tools_used=self.tools_used,
                                      recent_state_memory=self.state_memory[-2:])

            if len(self.state_memory) > self.window_size:
                if self.memory_mode == "sliding_window":
                    self.get_sliding_window_memory()
                    logger_iter.debug(f"Memory after sliding window applied: {len(self.state_memory)} messages.")
                    logger_iter.debug(f" System message:{self.state_memory[0]}")
                elif self.memory_mode == "summary":
                    self.get_summary_memory()
                    logger_iter.debug(f"Memory after summary applied: {len(self.state_memory)} messages.")
                    logger_iter.debug(f" Summary message:{self.state_memory[1]}")

            # --- THINK ---
            # The model reasons about what to do next
            response = ollama.chat(
                model=self.llm_model,
                messages=self.state_memory,
                options={"temperature": self.temperature},
                tools=[web_search]
            )

            message = response["message"]
            tool_calls = message.get("tool_calls", [])

            # --- ACT ---
            # The model decides to call tools or provide a final answer
            if tool_calls:

                self.state_memory.append({
                    "role": "assistant",
                    "content": message.get("content", ""),
                    "tool_calls": tool_calls
                })

                for call in tool_calls:
                    tool_name = call.function.name
                    tool_args = call.function.arguments
                    logger_iter.info(f"ACT  → calling '{tool_name}' with args {tool_args}")

                    # --- OBSERVE ---
                    # Run the tool and feed the result back
                    if tool_name == "web_search":
                        query = tool_args.get("query", "")
                        response = web_search(query=query, max_results=1)
                        observation = "Web search results:\n"
                        
                        for k, res in enumerate(response["results"]):
                            logger_iter.info(f"Result {k+1}: Title: {res['title']}, URL: {res['url']}")
                            observation += f"Content: {res['content']}\n\n"
                    else:
                        observation = f"Unknown tool: {tool_name}"
                        logger_iter.warning(f"Unknown tool requested: {tool_name}")

                    self.tools_used.append(tool_name)
                    observation_summary = self.get_content_summary(content=observation)
                    logger_iter.info(f"Observation summary from web search")

                    self.state_memory.append({
                        "role": "tool",
                        "tool_name": tool_name,
                        "content": observation_summary
                    })

            else:
                # if no tool calls, model provides the final answer 
                final_answer = message.get("content", "")
                logger_iter.info(f"Final answer reached after {iterations} iteration(s).")
                logger_iter.info(f"Final answer: {final_answer}")
                return final_answer

        # max iterations hit without a conclusive answer
        logger_iter.warning(f"Max iterations ({self.max_iterations}) reached without final answer.")
        return "I was unable to reach a final answer within the allowed number of steps."

In [23]:
PROMPT_SYSTEM = "You are a helpful assistant getting real-time information and summarizing it. You can use the web search tool to find up-to-date information when needed."

In [24]:
memory_agent = StatefullAgent(system=PROMPT_SYSTEM)    

In [25]:
response = memory_agent("Is there any news related to Venice Football Team?")

2026-05-04 13:36:47 | ACT  → calling 'web_search' with args {'query': 'Venice Football Team news Venezia FC 2026', 'max_results': 5}
2026-05-04 13:36:48 | Result 1: Title: Venezia FC strengthens its growth path: new commercial agreements with leading international partners, URL: https://www.veneziafc.it/en/news/venezia-fc-strengthens-its-growth-path-new-commercial-agreements-with-leading-international-partners
2026-05-04 13:36:48 | Result 2: Title: USMNT's Gianluca Busio helps Venezia return to Serie A - ESPN, URL: https://www.espn.co.uk/football/story/_/id/48648688/usa-gianluca-busio-venezia-promotion-serie-a
2026-05-04 13:36:48 | Result 3: Title: Venezia, presentato “Venezia Futuro”: Il primo passo nella formazione dei giovani calciatori - TUTTO mercato WEB, URL: https://www.tuttomercatoweb.com/serie-b/venezia-presentato-venezia-futuro-il-primo-passo-nella-formazione-dei-giovani-calciatori-2228132
2026-05-04 13:37:35 | Observation summary from web search
2026-05-04 13:37:55 | Final a

In [26]:
print("Final response:", response)

Final response: Here’s a quick snapshot of the latest headlines for **Venezia FC** (the Venice football club) as of early May 2026:

| Date | Headline | Key Take‑away |
|------|----------|---------------|
| **28 Apr 2026** | **Commercial & partnership deals** | The club’s ownership group inked multi‑year contracts with Elevate, Curva, Populous and CAA ICON to boost sponsorship, kit production and the new stadium project with the City of Venice. |
| **28 Apr 2026** | **Promotion to Serie A** | After a 2‑2 draw with Spezia, Venezia secured automatic promotion to Italy’s top flight for the 2026‑27 season, ending a one‑season stint in Serie B. |
| **27 Apr 2026** | **Youth development launch** | Venezia unveiled “Venezia Futuro,” a structured program for ages 8‑12 aimed at nurturing local talent and strengthening community ties. |
| **27 Apr 2026** | **Player spotlight** | USMNT midfielder Gianluca Busio, who joined in 2021, played a key role in the promotion campaign and will return to Se